# PROYECTO: Flujo ETL en Python con datos abiertos

In [2]:
# Librerías a usar
import requests
import pandas as pd

## Fase E (extraer)

En esta fase accedemos a la API y descargamos un archivo JSON con la información de las estaciones.

In [3]:
# Id de la app, PRIMARY KEY y endpoint
APP_ID = "api_bikepoint"
PRIMARY_KEY = "ad31f38c164d4bb2a2fb0d2d57d20d36"
ENDPOINT = "https://api.tfl.gov.uk/BikePoint/"

# Parámetros para el request
params = {
    "app_id": APP_ID,
    "app_key": PRIMARY_KEY
}

# Enviar request a la API
try:
    response = requests.get(ENDPOINT, params=params)

    # Mostrar excepción si hay algún error
    response.raise_for_status()

    # Si no hay error extraer los datos en formato JSON
    data = response.json()

    # Imprimir mensaje de confirmación
    print(f"Se descargaron correctamente datos de {len(data)} estaciones")

# Si lo anterior no funciona imprimir error
except requests.exceptions.RequestException as e:
    print(f"Error descargando los datos: {e}")

Se descargaron correctamente datos de 797 estaciones


## Fase T (Transformar)

En este punto los datos JSON están almacenados en una lista ("data").

Sin embargo, nos interesa extraer sólo la información relevante y ponerla en el formato adecuado. Esta será la fase de transformación.

Veamos dónde encontrar esta información. La variable "data" es un listado donde cada elemento es una estación:

In [3]:
len(data)

797

Cada estación es un diccionario. Veamos por ejemplo la primera estación:

In [4]:
data[0]

{'$type': 'Tfl.Api.Presentation.Entities.Place, Tfl.Api.Presentation.Entities',
 'id': 'BikePoints_1',
 'url': '/Place/BikePoints_1',
 'commonName': 'River Street , Clerkenwell',
 'placeType': 'BikePoint',
 'additionalProperties': [{'$type': 'Tfl.Api.Presentation.Entities.AdditionalProperties, Tfl.Api.Presentation.Entities',
   'category': 'Description',
   'key': 'TerminalName',
   'sourceSystemKey': 'BikePoints',
   'value': '001023',
   'modified': '2025-09-15T20:02:45.6Z'},
  {'$type': 'Tfl.Api.Presentation.Entities.AdditionalProperties, Tfl.Api.Presentation.Entities',
   'category': 'Description',
   'key': 'Installed',
   'sourceSystemKey': 'BikePoints',
   'value': 'true',
   'modified': '2025-09-15T20:02:45.6Z'},
  {'$type': 'Tfl.Api.Presentation.Entities.AdditionalProperties, Tfl.Api.Presentation.Entities',
   'category': 'Description',
   'key': 'Locked',
   'sourceSystemKey': 'BikePoints',
   'value': 'false',
   'modified': '2025-09-15T20:02:45.6Z'},
  {'$type': 'Tfl.Api.Pr

El nombre de la estación está en el campo "commonName":

In [5]:
data[0]["commonName"]

'River Street , Clerkenwell'

Mientras que su ubicación está en los campos 'lat' y 'lon':

In [6]:
print(data[0]["lat"])
print(data[0]["lon"])

51.529163
-0.10997


Los demás elementos de interés están en el campo "AdditionalProperties":

In [7]:
data[0]['additionalProperties'] # Acá están los campos de interés

[{'$type': 'Tfl.Api.Presentation.Entities.AdditionalProperties, Tfl.Api.Presentation.Entities',
  'category': 'Description',
  'key': 'TerminalName',
  'sourceSystemKey': 'BikePoints',
  'value': '001023',
  'modified': '2025-09-09T21:08:56.093Z'},
 {'$type': 'Tfl.Api.Presentation.Entities.AdditionalProperties, Tfl.Api.Presentation.Entities',
  'category': 'Description',
  'key': 'Installed',
  'sourceSystemKey': 'BikePoints',
  'value': 'true',
  'modified': '2025-09-09T21:08:56.093Z'},
 {'$type': 'Tfl.Api.Presentation.Entities.AdditionalProperties, Tfl.Api.Presentation.Entities',
  'category': 'Description',
  'key': 'Locked',
  'sourceSystemKey': 'BikePoints',
  'value': 'false',
  'modified': '2025-09-09T21:08:56.093Z'},
 {'$type': 'Tfl.Api.Presentation.Entities.AdditionalProperties, Tfl.Api.Presentation.Entities',
  'category': 'Description',
  'key': 'InstallDate',
  'sourceSystemKey': 'BikePoints',
  'value': '1278947280000',
  'modified': '2025-09-09T21:08:56.093Z'},
 {'$type':

Este campo es a su vez una lista de Python que contiene múltiples diccionarios, cada uno con los elementos de interés.

Veamos cuáles de estos campos podrían resultar interesantes. Imprimamos cada elemento con su "key" y "value" correspondientes:

In [8]:
for item in data[0]['additionalProperties']:
    print(f"key: {item['key']} - value: {item['value']}")

key: TerminalName - value: 001023
key: Installed - value: true
key: Locked - value: false
key: InstallDate - value: 1278947280000
key: RemovalDate - value: 
key: Temporary - value: false
key: NbBikes - value: 6
key: NbEmptyDocks - value: 8
key: NbDocks - value: 19
key: NbStandardBikes - value: 2
key: NbEBikes - value: 4


Y acá vemos estos campos de interés:

- Locked: indica si la estación está fuera de servicio (True) o en funcionamiento (false)
- NBikes: número total de bicicletas disponibles
- NbEmptyDocks: puestos para bicicletas disponibles
- NbDocks: número total de puestos para bicicletas
- NbStandardBikes: número total de bicicletas estándar disponibles
- NbEBikes: número total de bicicletas eléctricas disponibles

Estos son los campos que nos interesan. Así que con esta información ya podemos iterar por todas las estaciones y extraer la información de interés:

In [6]:
# Diccionario donde se almacenará la información de cada estación
info_stations = {
    'nombre_estacion': [], # stationName
    'disponible': [], # Locked
    'nBicis': [], # NbBikes
    'nEspaciosDisp': [], # nbEmptyDocks
    'nEspacios': [], # NbDocks
    'nBicisEstandar': [], # NbStandardBikes
    'nBicisElectricas': [], # NbEBikes
    'latitud': [],
    'longitud': [],
    # 'ultima_actualizacion': [], # modified 
}


for station in data:
    # Almacenar nombre de la estación, latitud y longitud
    info_stations['nombre_estacion'].append(station['commonName'])
    info_stations['latitud'].append(station['lat'])
    info_stations['longitud'].append(station['lon'])

    
    # Iterar por el key "additionalProperties" y extraer info actualizada de la estación
    for item in station['additionalProperties']:
        if item['key'] == 'NbBikes':
            info_stations['nBicis'].append(int(item['value']))

            # # Extraer marca de tiempo última actualización
            # datetime_update = pd.to_datetime(item['modified'])

            # # Preservar únicamente hasta horas, minutos y segundos
            # datetime_update = datetime_update.strftime("%Y-%m-%d %H:%M:%S")
            # info_stations['ultima_actualizacion'].append(datetime_update)
        if item['key'] == 'NbEmptyDocks':
            info_stations['nEspaciosDisp'].append(int(item['value']))
        if item['key'] == 'NbDocks':
            info_stations['nEspacios'].append(int(item['value']))
        if item['key'] == 'NbStandardBikes':
            info_stations['nBicisEstandar'].append(int(item['value']))
        if item['key'] == 'NbEBikes':
            info_stations['nBicisElectricas'].append(int(item['value']))
        if item['key'] == 'Locked':
            if item['value']=="false":
                info_stations['disponible'].append('Si')
            else:
                info_stations['disponible'].append('No')

# A DataFrame de Pandas
df = pd.DataFrame(info_stations)
df

,nombre_estacion,disponible,nBicis,nEspaciosDisp,nEspacios,nBicisEstandar,nBicisElectricas,latitud,longitud
0,"River Street , Clerkenwell",Si,5,12,19,5,0,51.529163,-0.109970
1,"Phillimore Gardens, Kensington",Si,31,5,37,31,0,51.499606,-0.197574
2,"Christopher Street, Liverpool Street",Si,24,7,32,17,7,51.521283,-0.084605
3,"St. Chad's Street, King's Cross",Si,16,2,23,11,5,51.530059,-0.120973
4,"Sedding Street, Sloane Square",Si,18,6,27,17,1,51.493130,-0.156876
...,...,...,...,...,...,...,...,...,...
792,"London Fields, Hackney",Si,2,16,21,2,0,51.541190,-0.058826
793,"Ashley Place, Victoria",Si,5,18,25,5,0,51.496160,-0.140947
794,"Portland Place, Marylebone",Si,4,28,34,3,1,51.520715,-0.145211
795,"Broadcasting House, Marylebone",Si,15,3,18,12,3,51.518117,-0.144228


## Fase L (carga)

Esta fase depende del uso final que queramos dar a los datos (visualización, alimentar un modelo, etc.).

De momento asumiremos que más adelante (en un próximo proyecto) nos enfocaremos en la visualización de los datos (algo como un *Dashboard*).

Así que podemos almacenarlos en formato Parquet (muy común en este tipo de proyectos):

In [18]:
df.to_parquet('estaciones.parquet')